# The mass law in the grid world

Settles the findings of `03_sigreg_mass.ipynb` and `02_grid_benchmark.ipynb` in one controlled world. The working rule: **a content
class is held in a latent of size D only if its share m of the pixel variance satisfies m · D ≥ κ** (κ ≈ 1.5). The share is the variance
over clips that the class contributes to the frames: a component that looks the same in every clip contributes nothing.

**Settings for every run.** SIGReg weight 0.01 (live encoder and shadow alike); random map; frame 48 × 48: a 12 × 8 map and 4 HUD rows
(16 inventory slots, up to 16 hand tiles, up to 16 compass tiles); one seed per cell, the last cell repeats steps 1, 4, 5 and 7 for two more
seeds (optional). Probes: a standardised MLP per part (inventory, position, facing, hand, mob, twinkle) of the latent and, for B, of its
shadow, plus the effective rank; read once at the end of every run. Kept = MLP R² ≥ 0.5, lost = R² < 0.2.

**The models**
- `detached`: one encoder trained by SIGReg alone (a predictor trains on its detached latents and is not read). Availability from mass alone.
- `ae`: the same encoder plus a decoder, trained by pixel reconstruction and SIGReg, no prediction. This is exactly how B's shadow encoder
  is trained (its predictor sees detached latents), here on its own.
- step 4 only: `sg` (prediction + SIGReg), A = `target` with a head start of 3000 steps, B = `retain_open`.

| step | question | runs |
|---|---|---|
| 0 | the mass of every component, the threshold κ/D per D | none (CPU) |
| 1 | does what is held follow the hand's mass? (hand shown in k = 1, 2, 4, 8, 16 tiles, D = 32) | 10 |
| 2 | does the threshold fall as 1/D? (D = 64 and 128; the `ae` also with the hand's contrast lowered, masses 0.1–0.4%) | 26 |
| 3 | does frequency count like size? (`ae`, at the mass where it crosses, 1.75%: 2 tiles always, or 4, 8, 16 tiles in 1/2, 1/4, 1/8 of the clips) | 3 |
| 5 | is location the barrier for the facing? (`ae`; the facing in a fixed compass of c = 1, 2, 4 tiles against the moving marker) | 3 |
| 4 | the recipe: the hand below, at and above the `ae`'s crossing (k = 1, 4, 16), `sg`, A and B, 20 000 steps | 9 |
| 7 | class-balanced reconstruction: the `ae` with each tile's error weighted by its ground-truth class's share^−p (p = 0.5, 1) | 4 |

**Predictions, written before any run** (steps 2, 3, 4 and 5 were adapted after step 1; the changes are marked below the step-1 table)
- **1:** the hand's readout rises with its mass and crosses 0.5 near κ/32 = 4.7% (between k = 4 and 8).
- **2:** the crossing mass falls about as 1/D: the slope of log(crossing) against log(D) lies between −1.3 and −0.7. Both models.
- **3:** the three equal-mass settings agree within 0.15 (readout on the clips where the hand is shown).
- **5:** the compass is held from about the threshold, as the hand; the moving marker is not held at the same mass (c = 2).
- **4:** below the threshold all three lose the hand. Above it `sg` loses it, A keeps it, B keeps it and drops the noise (mob, twinkle ≤ 0.3).
  If A is within 0.15 of B, mass replaces most of the shadow.
- **7:** facing and hand reach 0.5 at the default masses; the noise readouts rise with them (the risk of the weighting).

Run order: 0, 1, 2, 3, 5, 4, 7. Budget (step 1 took 12.5 minutes on the L4 with 3 workers): step 2 about 30 minutes, 3, 5 and 7
about 15 minutes together, step 4 (9 runs of 20 000 steps) about an hour and a half. Every run is resumable: finished runs in `OUT` are skipped.

In [ ]:
%pip install -q -e ..
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from gaf.env_grid import GridCfg, component_masses
from gaf.train import Cfg
from gaf import analysis as A

OUT = "/workspace/results/mass_law"
SEED = 0
EXTRA_SEEDS = (1, 2)                  # the optional cell at the end
WORKERS = 3                           # runs at once on the GPU
KAPPA = 1.5

# ---- the knobs ---------------------------------------------------------------------------
FRAME = dict(W=12, H=8)               # 48 x 48 pixels: a 12 x 8 map and 4 HUD rows
K = (1, 2, 4, 8, 16)                  # tiles that show the hand (steps 1, 2)
DS = (32, 64, 128)                    # latent sizes (step 2; 32 is step 1)
CONTRAST = (0.7, 0.5, 0.35)           # step 2 (ae): the hand in 1 tile with its contrast lowered: masses 0.43, 0.22, 0.11 %
PRESENCE = ((4, 0.5), (8, 0.25), (16, 0.13))   # step 3: (tiles, share of the clips), each with the mass of 2 tiles always shown (1.75 %)
COMPASS = (1, 2, 4)                   # step 5: compass tiles (1.7, 3.3, 6.5 %)
RECIPE_K = (1, 4, 16)                 # step 4: the hand below (0.88 %), at (3.45 %) and above (12.5 %) the ae's crossing at D = 32
HEAD_START = 3000                     # step 4: variant A
CB_POWERS, CB_K = (0.5, 1.0), (1, 2)  # step 7
CLUSTERS = (16, 32, 64)               # step 8: appearance clusters of the label-free classes (x 6 position types)
B_STEPS = 12_000                      # step 8b: B end to end
PARTS = ["hand", "facing", "mob", "twinkle", "inventory", "position"]

LEAN = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=5000, floor_start=10 ** 9, eval_every=1000)   # no gate: no floor probes
FULL = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=20_000, basis="excess", floor_steps=1500, floor_warm=True)
pd.set_option("display.precision", 3)


def grid(**kw):
    return GridCfg(**{**FRAME, **kw})


def lean(arm, env, tag, D=32, seed=SEED, **kw):
    return Cfg(arm=arm, seed=seed, D=D, env=env, tag=tag, **{**LEAN, **kw})


def full(arm, env, tag, seed=SEED, **kw):
    return Cfg(arm=arm, seed=seed, D=32, env=env, tag=tag, **{**FULL, **kw})


def pick(df, seeds=(SEED,), **sel):
    d = A.select(df, **sel)
    return d[d.seed.isin(seeds)]


def crossing(masses, readouts, thr=0.5):
    """The mass at which the readout reaches thr, log-linear between the swept points; (value, note)."""
    m, r = np.asarray(masses, float), np.asarray(readouts, float)
    hit = np.where(r >= thr)[0]
    if not len(hit):
        return np.nan, f"> {m[-1]:.2f}%"
    i = hit[0]
    if i == 0:
        return np.nan, f"< {m[0]:.2f}%"
    f = (thr - r[i - 1]) / (r[i] - r[i - 1])
    v = float(np.exp(np.log(m[i - 1]) + f * (np.log(m[i]) - np.log(m[i - 1]))))
    return v, f"{v:.2f}%"

## 0 The masses
Each component's share of the pixel variance in this frame (2048 clips, CPU, about a minute), the hand at every k, the compass at every c,
and the threshold κ/D. The predicted crossing: the first k whose mass reaches κ/D.

In [ ]:
base = component_masses(grid(), n=2048)
display(pd.Series({k: round(100 * v, 2) for k, v in base.items()}, name="% of the pixel variance").to_frame())
MASS = {k: 100 * component_masses(grid(hand_tiles=k), n=2048)["hand"] for k in K}
CMASS = {c: 100 * component_masses(grid(compass_tiles=c), n=2048)["compass"] for c in COMPASS}
PMASS = {(k, p): 100 * component_masses(grid(hand_tiles=k, hand_p=p), n=2048)["hand"] for k, p in PRESENCE}
SMASS = {s: 100 * component_masses(grid(hand_contrast=s), n=2048)["hand"] for s in CONTRAST}
MARKER = 100 * base["facing marker"]
display(pd.DataFrame({"hand mass % (contrast, 1 tile)": SMASS}).T.round(3))
display(pd.DataFrame({"hand mass %": MASS}).T.round(2))
display(pd.DataFrame({"compass mass %": CMASS}).T.round(2))
display(pd.DataFrame({"hand mass % (with presence)": {f"k={k}, p={p}": v for (k, p), v in PMASS.items()}}).T.round(2))
pred = {}
for D in DS:
    tau = 100 * KAPPA / D
    first = next((k for k in K if MASS[k] >= tau), None)
    pred[D] = {"threshold %": round(tau, 2), "first k at or above": first}
display(pd.DataFrame(pred).T)

## 1 The hand's mass at D = 32
`detached` and `ae`, hand shown in k = 1, 2, 4, 8, 16 tiles, 5000 steps. The table gives every part's readout (MLP, standardised) and the
effective rank; the facing marker, mob and twinkle do not change with k and show what else each model holds.

In [ ]:
E1 = [lean(arm, grid(hand_tiles=k), f"k{k}") for arm in ("detached", "ae") for k in K]
A.execute(E1, OUT, workers=WORKERS)
df = A.collect(OUT)


def sweep(df, arm, D, seeds=(SEED,)):
    rows = {}
    for k in K:
        r = pick(df, seeds, arm=arm, tag=f"k{k}", D=D)
        if len(r):
            rows[f"k={k} ({MASS[k]:.2f}%)"] = r[[f"pm_{p}" for p in PARTS] + ["pm_rank"]].mean()
    return pd.DataFrame(rows).T


for arm in ("detached", "ae"):
    print(arm, "D = 32")
    display(sweep(df, arm, 32).round(3))

**Result of step 1 (first pod run, seed 0), and what it changed**

| D = 32 | hand at 0.88 / 1.75 / 3.45 / 6.66 / 12.5 % | facing marker, mob, position (moving content) | inventory |
|---|---|---|---|
| `detached` | −0.07, −0.08, −0.02, 0.03, **0.25** | all ≤ 0 (−0.08 … −0.14) | 0 – 0.16 |
| `ae` | −0.09, **0.50**, 0.98, 0.96, 0.98 | all ≤ 0 | 0.6 – 0.8 |

- The `ae` crosses 0.5 at about **1.75 %** (m · D = 0.56), not at the predicted 4.7 % (κ = 1.5). `detached` does not cross within the sweep
  (m · D > 4). So κ is not one constant: reconstruction lowers the threshold at least about seven-fold against SIGReg alone.
- **Content that moves on the map is held by neither model** at any mass (position of the agent, 10 %; mob, 3.7 %; facing marker, 3.0 %), while
  fixed-location content is (hand, inventory). The mass rule holds for fixed-location content only; step 5 tests the facing in a fixed place.
- Effective rank is full (30.9 – 31.6 of 32) everywhere.

*Adapted after step 1 (before steps 2 – 5), the pre-registered predictions above stay as they were written:* step 2 runs D = 64 and 128
only, and the `ae` also gets hand tiles of lowered contrast (0.1 – 0.4 %), because its crossing at D = 64 and 128 is predicted near 0.9 % and
0.4 % if m · D stays constant (κ ≈ 0.56), below the one-tile mass; `detached` keeps the k sweep. Step 3 runs the `ae` at its crossing mass
(1.75 %), where frequency and size can be told apart (at 3.45 % it is saturated at 0.98). Step 5 runs the `ae` with the compass at 1.7, 3.3 and
6.5 %. Step 4 uses k = 1, 4 and 16: below the `ae`'s crossing (the shadow holds nothing), at 3.45 % (the shadow holds the hand, `detached` does
not), and at 12.5 % (`detached` starts to). Revised predictions: **2:** `ae` crossing × D roughly constant (slope −1) with κ ≈ 0.56; `detached`
crossing × D > 4 (slope −1 from the values that are bracketed). **4:** at k = 4, B keeps the hand and `sg` and A lose it; at k = 16, A is
partly there (`detached` 0.25 at D = 32); at k = 1 all lose it.

## 2 The threshold against D
The same sweep at D = 64 and 128 (D = 32 is step 1). For the `ae` the sweep also has hand tiles of lowered contrast (0.1 – 0.4 %), below
one tile. 26 runs. For every model and D: the mass at which the hand's readout crosses 0.5, and the slope of log(crossing) against log(D)
(the rule: −1); the product crossing × D is κ.

In [ ]:
E2 = [lean(arm, grid(hand_tiles=k), f"k{k}", D=D) for arm in ("detached", "ae") for D in DS if D != 32 for k in K]
E2 += [lean("ae", grid(hand_contrast=s), f"s{s}", D=D) for D in DS if D != 32 for s in CONTRAST]
A.execute(E2, OUT, workers=WORKERS)
df = A.collect(OUT)
POINTS = {"detached": [(MASS[k], f"k{k}") for k in K],
          "ae": sorted([(MASS[k], f"k{k}") for k in K] + [(SMASS[s], f"s{s}") for s in CONTRAST])}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True, layout="constrained")
CROSS = {}
for ax, arm in zip(axes, ("detached", "ae")):
    for D, color in zip(DS, A.SERIES):
        rows = [(m_, pick(df, arm=arm, tag=tag, D=D)["pm_hand"].mean()) for m_, tag in POINTS[arm]]
        rows = [(m, r) for m, r in rows if not np.isnan(r)]
        if not rows:
            continue
        m, r = zip(*rows)
        ax.plot(m, r, marker="o", color=color, label=f"D={D}")
        ax.axvline(100 * KAPPA / D, color=color, ls="--", lw=1)
        CROSS[(arm, D)] = crossing(m, r)
    ax.set(xscale="log", title=arm, xlabel="hand's share of the pixel variance (%)", ylim=(-0.05, 1.02))
axes[0].set_ylabel("MLP readout of the hand")
axes[1].legend(fontsize=8)
t = pd.DataFrame({arm: {D: CROSS.get((arm, D), (np.nan, "-"))[1] for D in DS} for arm in ("detached", "ae")})
t["rule at kappa = 1.5"] = [f"{100 * KAPPA / D:.2f}%" for D in DS]
display(t)
print("kappa = crossing x D (the rule: constant across D)")
display(pd.DataFrame({arm: {D: CROSS[(arm, D)][0] * D / 100 for D in DS if (arm, D) in CROSS and not np.isnan(CROSS[(arm, D)][0])}
                      for arm in ("detached", "ae")}).round(2))
SLOPE = {}
for arm in ("detached", "ae"):
    pts = [(D, CROSS[(arm, D)][0]) for D in DS if (arm, D) in CROSS and not np.isnan(CROSS[(arm, D)][0])]
    SLOPE[arm] = np.polyfit(np.log([p[0] for p in pts]), np.log([p[1] for p in pts]), 1)[0] if len(pts) >= 2 else np.nan
    print(f"{arm}: slope of log(crossing) against log(D) = {SLOPE[arm]:.2f} from {len(pts)} bracketed crossings (rule: -1)")

**Reading 1–2.** A rising readout with a crossing near κ/D and a slope near −1: the rule holds for discrete, fixed-location content, and
κ can be read off (crossing × D). A crossing that does not move with D: the limit is not the latent's size, and the rule's D is the wrong variable.
`ae` crossing at a lower mass than `detached`: reconstruction lowers the threshold (it adds a reason to keep content beyond its mass).

## 3 Frequency against size
The `ae` at the mass where it crosses (1.75 %, step 1's k = 2 run reads 0.50): the hand in 4 tiles in half of the clips, in 8 tiles in a
quarter, in 16 tiles in an eighth, each with the mass of 2 tiles always shown. (At 3.45 % the `ae` is saturated at 0.98 and could not tell the
settings apart.) Hidden tiles show the average hand colour, as an absent block in Craftax shows ordinary terrain. The readout is taken on the
clips where the hand is shown (`pm_hand_visible`). `detached` is not run here: it holds nothing at these masses.

In [ ]:
E3 = [lean("ae", grid(hand_tiles=k, hand_p=p), f"p{p}-k{k}") for k, p in PRESENCE]
A.execute(E3, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for arm in ("ae",):
    r = pick(df, arm=arm, tag="k2", D=32)
    if len(r):
        rows[(arm, f"2 tiles, always ({MASS[2]:.2f}%)")] = {"hand readout": r["pm_hand"].mean(), "rank": r["pm_rank"].mean()}
    for k, p in PRESENCE:
        r = pick(df, arm=arm, tag=f"p{p}-k{k}", D=32)
        if len(r):
            rows[(arm, f"{k} tiles, {int(100 * p)}% of clips ({PMASS[(k, p)]:.2f}%)")] = {
                "hand readout": r["pm_hand_visible"].mean(), "rank": r["pm_rank"].mean()}
T3 = pd.DataFrame(rows).T
display(T3.round(3))

## 5 The facing in a fixed place
The facing shown in a compass of c = 1, 2, 4 HUD tiles (1.7, 3.3, 6.5 %; c = 2 has about the marker's mass, 3.0 %), the `ae`, D = 32. Compared
with the marker alone (step 1, k = 1: no compass) and with the hand at similar masses (step 1). The fixed-location hand is held from 1.75 %;
the moving marker at 3.0 % is not. `detached` is not run: it holds nothing at these masses.

In [ ]:
E5 = [lean("ae", grid(compass_tiles=c), f"c{c}") for c in COMPASS]
A.execute(E5, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for arm in ("ae",):
    r = pick(df, arm=arm, tag="k1", D=32)
    if len(r):
        rows[(arm, f"marker only ({MARKER:.2f}%)")] = {"facing readout": r["pm_facing"].mean()}
    for c in COMPASS:
        r = pick(df, arm=arm, tag=f"c{c}", D=32)
        if len(r):
            rows[(arm, f"compass c={c} ({CMASS[c]:.2f}%)")] = {"facing readout": r["pm_facing"].mean()}
    for k in (2, 4):
        r = pick(df, arm=arm, tag=f"k{k}", D=32)
        if len(r):
            rows[(arm, f"hand k={k} ({MASS[k]:.2f}%)")] = {"facing readout": np.nan, "hand readout": r["pm_hand"].mean()}
T5 = pd.DataFrame(rows).T
display(T5.round(3))

**Reading 3 and 5.** Step 3: equal readouts mean frequency counts like size, so a rare class can be lifted by sampling; a lower readout for
the rare settings means rarity costs more than its mass says. Step 5: a compass held like the hand at the same mass while the marker is not
means the moving location is the barrier for the facing, not its mass.

## 4 The recipe
The hand in k = 1 tile (0.88 %, below the `ae`'s crossing), 4 tiles (3.45 %, where the `ae` holds it and `detached` does not) and 16 tiles (12.5 %,
where `detached` starts to). `sg`, A = `target` with a head start of 3000 steps (the encoder sees only SIGReg while the predictor learns),
B = `retain_open`. 20 000 steps, with the floor gate on. The table reads the live
encoder (the model that predicts) and, for B, its shadow; the settledness check looks at the last 4000 steps.

In [ ]:
E4 = [full(arm, grid(hand_tiles=k), f"recipe-k{k}", head_start=HEAD_START if arm == "target" else 0)
      for k in RECIPE_K for arm in ("sg", "target", "retain_open")]
A.execute(E4, OUT, workers=WORKERS)
df = A.collect(OUT)
NAME = {"sg": "sg", "target": "A: target + head start", "retain_open": "B: retain_open"}
rows, runs = {}, {}
for k in RECIPE_K:
    for arm in ("sg", "target", "retain_open"):
        r = pick(df, arm=arm, tag=f"recipe-k{k}")
        if len(r):
            row = {f"live {p}": r[f"pm_{p}"].mean() for p in PARTS}
            if arm == "retain_open":
                row.update({f"shadow {p}": r[f"sh_pm_{p}"].mean() for p in ("hand", "facing", "mob")})
            rows[(f"k={k} ({MASS[k]:.2f}%)", NAME[arm])] = row
            runs[f"k={k} {NAME[arm]}"] = dict(arm=arm, tag=f"recipe-k{k}", seed=SEED)
T4 = pd.DataFrame(rows).T
display(T4.round(3))
print("settled? change and swing over the last 4000 steps (ridge readouts of the eval rows)")
display(A.drift(df, OUT, runs, cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow")))

**Reading 4.** At k = 4: B high and `sg` and A low means the shadow supplies availability that mass alone (SIGReg alone) does not, so the
shadow is needed there; A high as well means a head start does the job. At k = 16: A within 0.15 of B means mass has replaced the shadow's
availability job, and B is needed only for noise (compare the mob and twinkle columns). B high at k = 1 would contradict the `ae`'s crossing.
Unsettled runs are provisional.

## 7 Class-balanced reconstruction
The `ae` with each tile's reconstruction error weighted by its ground-truth tile class's share of the variance to the power −p (p = 0.5
and 1; capped at 30×, mean weight 1), at the default masses (hand k = 1 and 2). Compared with the plain `ae` of step 1. The noise columns
show the risk: rare noise tiles (the mob) are lifted too.

In [ ]:
E7 = [lean("ae", grid(hand_tiles=k), f"cb{pw}-k{k}", rec_class_power=pw) for pw in CB_POWERS for k in CB_K]
A.execute(E7, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for k in CB_K:
    for pw, tag in [(0.0, f"k{k}")] + [(pw, f"cb{pw}-k{k}") for pw in CB_POWERS]:
        r = pick(df, arm="ae", tag=tag, D=32)
        if len(r):
            rows[(f"k={k} ({MASS[k]:.2f}%)", f"power {pw}")] = {**{p: r[f"pm_{p}"].mean() for p in PARTS},
                                                               "rank": r["pm_rank"].mean(), "rec mse": r["ae_rec_mse"].mean()}
T7 = pd.DataFrame(rows).T
display(T7.round(3))

## 8 Label-free class weights
Step 7 weights each tile by the share of its ground-truth class. Here the classes come from the pixels alone: k-means clusters of the tile
appearance (k = 16, 32, 64), crossed with six *position types* (k-means over the tile positions by how often the position changes between
frames, and its pixel spread). The position type is needed because the hand tile looks exactly like a map tile: appearance alone puts it
in the terrain clusters (weight about that of the floor); the change rate (hand 47 %, map 3 %) isolates it. The control `appearance only`
(1 position type) should therefore fail like the plain `ae`. Everything else as in step 7 (power 1, hand k = 1, 0.88 %).

**8a** the `ae` (like step 7). **8b** B end to end (the working method, 12k steps, live encoder) with: no weights, ground-truth weights,
cluster weights. Prediction: clusters match the ground-truth weights on hand and position; the mob and twinkle columns show whether the
unpredictable noise gets lifted too (the missing predictability guard).

In [ ]:
E8 = ([lean("ae", grid(hand_tiles=1), f"cl{c}", rec_class_power=1.0, rec_clusters=c) for c in CLUSTERS]
      + [lean("ae", grid(hand_tiles=1), "cl32-apponly", rec_class_power=1.0, rec_clusters=32, rec_pos_types=1)])
A.execute(E8, OUT, workers=WORKERS)
df = A.collect(OUT)
sel = [("plain", "k1"), ("ground truth", "cb1.0-k1")] + [(f"clusters {c} x 6 types", f"cl{c}") for c in CLUSTERS] \
      + [("appearance only (32)", "cl32-apponly")]
rows = {}
for name, tag in sel:
    r = pick(df, arm="ae", tag=tag, D=32)
    if len(r):
        rows[name] = {**{p: r[f"pm_{p}"].mean() for p in PARTS}, "rank": r["pm_rank"].mean(), "rec mse": r["ae_rec_mse"].mean()}
T8a = pd.DataFrame(rows).T
display(T8a.round(3))

In [ ]:
E8b = [full("retain_open", grid(hand_tiles=1), tag, steps=B_STEPS, **kw)
       for tag, kw in [("b-plain", {}), ("b-gt", dict(rec_class_power=1.0)),
                       ("b-cl32", dict(rec_class_power=1.0, rec_clusters=32))]]
A.execute(E8b, OUT, workers=WORKERS)
df = A.collect(OUT)
rows, runs = {}, {}
for name, tag in [("B plain", "b-plain"), ("B + ground truth", "b-gt"), ("B + clusters (32 x 6)", "b-cl32")]:
    r = pick(df, arm="retain_open", tag=tag, D=32)
    if len(r):
        runs[name] = dict(arm="retain_open", tag=tag, seed=SEED)
        rows[name] = {**{f"live {p}": r[f"pm_{p}"].mean() for p in PARTS}, **{f"shadow {p}": r[f"sh_pm_{p}"].mean() for p in ("hand", "position", "inventory")}}
T8b = pd.DataFrame(rows).T
display(T8b.round(3))
print("settled? change and swing over the last 4000 steps (ridge readouts of the eval rows)")
display(A.drift(df, OUT, runs, cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow")))

**Reading 8.** The labels were only a stand-in for "rare content": the clusters work if 8a reaches the ground-truth column (hand about 0.9,
position about 0.7) and 8b follows. `appearance only` low confirms the position type is what finds the hand. If the noise columns (mob,
twinkle) rise above the ground-truth run, the unpredictable tiles are being lifted: the next step is the predictability guard (down-weight
the clusters the shadow predictor cannot predict). Unsettled runs are provisional.

## Verdicts
The rules above, applied to whatever has run.

In [ ]:
def say(label, ok):
    print(f"{label}: {'holds' if ok else 'does not hold'}")


try:
    s1 = {arm: sweep(df, arm, 32)["pm_hand"] for arm in ("detached", "ae")}
    for arm, s in s1.items():
        print(f"step 1, {arm}: hand readout by mass", s.round(2).to_dict())
        say(f"  rises with mass ({arm})", s.iloc[-1] - s.iloc[0] >= 0.3)
except Exception as e:  # noqa: BLE001
    print("step 1: not run yet", e)
if "SLOPE" in globals():
    for arm, v in SLOPE.items():
        say(f"step 2, {arm}: slope {v:.2f} within [-1.3, -0.7]", -1.3 <= v <= -0.7)
if "T3" in globals():
    for arm in ("ae",):
        v = T3.loc[arm, "hand readout"]
        say(f"step 3, {arm}: equal-mass settings within 0.15 (spread {v.max() - v.min():.2f})", v.max() - v.min() <= 0.15)
if "T5" in globals():
    for arm in ("ae",):
        t = T5.loc[arm]
        m = t.loc[t.index.str.startswith("marker"), "facing readout"]
        c2 = t.loc[t.index.str.startswith("compass c=2"), "facing readout"]
        if len(m) and len(c2):
            say(f"step 5, {arm}: compass c=2 ({c2.iloc[0]:.2f}) above the marker ({m.iloc[0]:.2f}) by 0.3", c2.iloc[0] - m.iloc[0] >= 0.3)
if "T4" in globals():
    lo, mid, hi = (f"k={k} ({MASS[k]:.2f}%)" for k in RECIPE_K)
    try:
        say("step 4, k=1: all three lose the hand (< 0.2)", (T4.loc[lo, "live hand"] < 0.2).all())
        h = T4.loc[mid, "live hand"]
        say("step 4, k=4: sg and A lose it (< 0.2)", h[NAME["sg"]] < 0.2 and h[NAME["target"]] < 0.2)
        say("step 4, k=4: B keeps it (>= 0.5)", h[NAME["retain_open"]] >= 0.5)
        h = T4.loc[hi, "live hand"]
        say("step 4, k=16: sg loses the hand (< 0.2)", h[NAME["sg"]] < 0.2)
        say("step 4, k=16: B keeps it (>= 0.5)", h[NAME["retain_open"]] >= 0.5)
        say("step 4, k=16: A within 0.15 of B", abs(h[NAME["target"]] - h[NAME["retain_open"]]) <= 0.15)
        say("step 4, B drops the noise at k=4 and 16 (mob, twinkle <= 0.3)",
            all(max(T4.loc[(g, NAME["retain_open"]), "live mob"], T4.loc[(g, NAME["retain_open"]), "live twinkle"]) <= 0.3 for g in (mid, hi)))
    except KeyError as e:
        print("step 4: not complete", e)
if "T7" in globals():
    for k in CB_K:
        t = T7.loc[f"k={k} ({MASS[k]:.2f}%)"]
        best = t.drop("power 0.0", errors="ignore")
        say(f"step 7, k={k}: facing and hand reach 0.5 with the weighting", ((best["facing"] >= 0.5) & (best["hand"] >= 0.5)).any())
if "T8a" in globals() and "ground truth" in T8a.index:
    cl = T8a.loc[[i for i in T8a.index if i.startswith("clusters")]]
    say("step 8a: a cluster setting matches the ground-truth hand readout within 0.15", (T8a.loc["ground truth", "hand"] - cl["hand"]).min() <= 0.15)
    say("step 8a: appearance only misses the hand (< 0.3)", T8a.loc["appearance only (32)", "hand"] < 0.3)
    say("step 8a: clusters lift the noise no more than 0.2 above ground truth (mob, twinkle)",
        (cl[["mob", "twinkle"]].max(axis=1) - T8a.loc["ground truth", ["mob", "twinkle"]].max()).min() <= 0.2)

## All seeds (optional)
Steps 1, 4, 5 and 7 again for `EXTRA_SEEDS`, then the seed means and spreads of the hand (and facing) readouts. About 2.5 times the time of
those steps.

In [ ]:
RUN_ALL_SEEDS = False
if RUN_ALL_SEEDS:
    extra = []
    for s in EXTRA_SEEDS:
        extra += [lean(arm, grid(hand_tiles=k), f"k{k}", seed=s) for arm in ("detached", "ae") for k in K]
        extra += [lean("ae", grid(compass_tiles=c), f"c{c}", seed=s) for c in COMPASS]
        extra += [full(arm, grid(hand_tiles=k), f"recipe-k{k}", seed=s, head_start=HEAD_START if arm == "target" else 0)
                  for k in RECIPE_K for arm in ("sg", "target", "retain_open")]
        extra += [lean("ae", grid(hand_tiles=k), f"cb{pw}-k{k}", seed=s, rec_class_power=pw) for pw in CB_POWERS for k in CB_K]
    A.execute(extra, OUT, workers=WORKERS)
    df = A.collect(OUT)
    seeds = (SEED, *EXTRA_SEEDS)
    d = df[df.seed.isin(seeds) & (df.D == 32)]
    for col in ("pm_hand", "pm_facing"):
        print(col, "(mean, sd over seeds)")
        display(d.pivot_table(index="tag", columns="arm", values=col, aggfunc=["mean", "std"]).round(3))

## All runs

In [ ]:
df = A.collect(OUT)
df.to_csv(f"{OUT}/summary.csv", index=False)
print(len(df), "runs;", f"{df.seconds.sum() / 3600:.1f} GPU-hours")